# 0824_peace_002_mapping_aware_xgboost

Siemens AOI 데이터의 5개 검사유형을 `mapping.json` 기준으로 마스킹한 뒤 하나의 XGBoost로 학습한다.

사전 등록된 핵심 원칙:

- 앞 70%에서만 3개 Walk-forward Fold를 수행해 중복 전략과 클래스 가중치를 선택한다.
- 70~80% Final Validation/Calibration은 최종 Early Stopping과 Recall 97% 임계값 선택에만 사용한다.
- 마지막 20% Final Test는 모든 결정이 고정되고 Walk-forward 안전성 gate를 통과한 경우에만 한 번 평가한다.
- Final Test 결과를 보고 모델, 피처, 가중치 또는 임계값을 다시 선택하지 않는다.


## 1. 설정과 라이브러리

실험 ID, 재현 시드, 라이브러리 버전과 입력·출력 경로를 고정한다. 데이터와 매핑 파일은 같은 디렉터리에 있는 쌍만 인정하며, 서로 다른 복사본이 발견되면 실행을 중단한다.


In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0824_peace_002_mapping_aware_xgboost"
RANDOM_STATE = 42
CALIBRATION_MIN_RECALL = 0.97
EVALUATION_MIN_RECALL = 0.95
BASELINE_TEST_PR_AUC = 0.236803

np.random.seed(RANDOM_STATE)

LIBRARY_VERSIONS = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "joblib": joblib.__version__,
}


def locate_project_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md와 notebooks/가 있는 프로젝트 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def locate_input_pair() -> tuple[Path, Path, list[dict]]:
    cwd = Path.cwd()
    candidate_directories = [
        cwd / "data" / "raw",
        cwd.parent / "data" / "raw",
        cwd,
        cwd.parent,
        cwd.parent.parent,
    ]
    candidates = []
    seen_pairs = set()
    for directory in candidate_directories:
        data_path = directory / "dataset.csv"
        mapping_path = directory / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            resolved_pair = (data_path.resolve(), mapping_path.resolve())
            if resolved_pair not in seen_pairs:
                seen_pairs.add(resolved_pair)
                candidates.append(resolved_pair)
    if not candidates:
        raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")

    audit = []
    for data_path, mapping_path in candidates:
        audit.append(
            {
                "data_path": str(data_path),
                "mapping_path": str(mapping_path),
                "data_sha256": sha256_file(data_path),
                "mapping_sha256": sha256_file(mapping_path),
            }
        )
    distinct_hash_pairs = {
        (row["data_sha256"], row["mapping_sha256"]) for row in audit
    }
    if len(distinct_hash_pairs) != 1:
        raise RuntimeError("서로 다른 SHA-256의 입력 복사본이 발견됐습니다.")
    return candidates[0][0], candidates[0][1], audit


PROJECT_ROOT = locate_project_root()
DATA_PATH, MAPPING_PATH, INPUT_AUDIT = locate_input_pair()
MODEL_PATH = PROJECT_ROOT / "models" / f"{EXPERIMENT_ID}.pkl"
DATA_SHA256 = sha256_file(DATA_PATH)
MAPPING_SHA256 = sha256_file(MAPPING_PATH)

print("experiment:", EXPERIMENT_ID)
print("project root:", PROJECT_ROOT)
print("dataset:", DATA_PATH)
print("mapping:", MAPPING_PATH)
print("model output:", MODEL_PATH)
display(pd.Series(LIBRARY_VERSIONS, name="version"))
display(pd.DataFrame(INPUT_AUDIT))


experiment: 0824_peace_002_mapping_aware_xgboost
project root: /Users/peace/Downloads/2026 여름방학 AI 제조 부캠/최종프로젝트/raw_dataset/siemens_aoi_ML_practice
dataset: /Users/peace/Downloads/2026 여름방학 AI 제조 부캠/최종프로젝트/raw_dataset/dataset.csv
mapping: /Users/peace/Downloads/2026 여름방학 AI 제조 부캠/최종프로젝트/raw_dataset/mapping.json
model output: /Users/peace/Downloads/2026 여름방학 AI 제조 부캠/최종프로젝트/raw_dataset/siemens_aoi_ML_practice/models/0824_peace_002_mapping_aware_xgboost.pkl


python          3.12.7
numpy           1.26.4
pandas           2.2.2
scikit_learn     1.5.1
xgboost          3.4.1
joblib           1.4.2
Name: version, dtype: object

,data_path,mapping_path,data_sha256,mapping_sha256
0,/Users/peace/Downloads/2026 여름방학 AI 제ᄌ...,/Users/peace/Downloads/2026 여름방학 AI 제ᄌ...,53e8568743216d556856ed69b388f6750fbfa0b8c59ad3...,3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a...


## 2. 데이터 로드와 필수 검증

원본 파일은 읽기만 한다. 첫 번째 저장 인덱스 열을 `record_id`로 명명하고, 문서에 확정된 행·라벨·피처·매핑 조건을 assert한다.


In [2]:
with MAPPING_PATH.open(encoding="utf-8") as stream:
    inspection_mapping = json.load(stream)

raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if str(source_index_column).startswith("Unnamed:"):
    raw_df = raw_df.rename(columns={source_index_column: "record_id"})
elif source_index_column != "record_id":
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

required_columns = {"record_id", "timestamp", "class", "inspection_type"}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"

raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], errors="raise", utc=True)
raw_df["class"] = raw_df["class"].astype("int8")
raw_df = raw_df.sort_values(["timestamp", "record_id"], kind="stable").reset_index(drop=True)


def inspection_feature_key(column: str) -> int:
    return int(column.removeprefix("inspection_feat"))


inspection_columns = sorted(
    [column for column in raw_df.columns if column.startswith("inspection_feat")],
    key=inspection_feature_key,
)
mapped_inspection_columns = sorted(
    set().union(*(set(columns) for columns in inspection_mapping.values())),
    key=inspection_feature_key,
)
globally_unmapped_features = sorted(
    set(inspection_columns) - set(mapped_inspection_columns),
    key=inspection_feature_key,
)
mapping_references = set().union(*(set(columns) for columns in inspection_mapping.values()))
numeric_input_columns = [
    column for column in raw_df.columns
    if column not in {"record_id", "timestamp", "class"}
]

assert len(raw_df) == 440_274
assert raw_df["record_id"].nunique() == len(raw_df)
assert set(raw_df["class"].unique()) == {0, 1}
assert int((raw_df["class"] == 1).sum()) == 4_622
assert int((raw_df["class"] == 0).sum()) == 435_652
assert set(raw_df["inspection_type"].unique()) == {0, 1, 2, 3, 4}
assert {int(key) for key in inspection_mapping} == {0, 1, 2, 3, 4}
assert mapping_references <= set(raw_df.columns)
assert raw_df["timestamp"].notna().all()
assert not np.isinf(raw_df[numeric_input_columns].to_numpy()).any()
assert len(inspection_columns) == 70
assert len(mapped_inspection_columns) == 65
assert len(globally_unmapped_features) == 5

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "timestamp_groups": raw_df["timestamp"].nunique(),
        "positive_rows": int(raw_df["class"].sum()),
        "positive_rate_pct": raw_df["class"].mean() * 100,
        "inspection_features": len(inspection_columns),
        "mapped_inspection_features": len(mapped_inspection_columns),
        "globally_unmapped_features": len(globally_unmapped_features),
        "start_time": raw_df["timestamp"].min(),
        "end_time": raw_df["timestamp"].max(),
    },
    name="verified_data",
)
display(data_summary)
print("globally unmapped:", globally_unmapped_features)


rows                                             440274
columns                                              78
timestamp_groups                                  39742
positive_rows                                      4622
positive_rate_pct                              1.049801
inspection_features                                  70
mapped_inspection_features                           65
globally_unmapped_features                            5
start_time                    1970-06-23 03:58:55+00:00
end_time                      1970-11-02 14:21:28+00:00
Name: verified_data, dtype: object

globally unmapped: ['inspection_feat14', 'inspection_feat15', 'inspection_feat35', 'inspection_feat36', 'inspection_feat37']


## 3. 시간순 경계와 Final holdout

중복을 제거하기 전에 누적 행 비율이 목표값에 가장 가까운 timestamp 그룹의 끝을 30/40/50/60/70/80% 경계로 사용한다. 같은 timestamp는 절대 두 구간으로 나누지 않는다.


In [3]:
BOUNDARY_FRACTIONS = (0.30, 0.40, 0.50, 0.60, 0.70, 0.80)
timestamp_group_sizes = raw_df.groupby("timestamp", sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_values = timestamp_group_sizes.index.to_numpy()


def closest_group_end(fraction: float) -> pd.Timestamp:
    target_rows = len(raw_df) * fraction
    position = int(np.abs(cumulative_rows - target_rows).argmin())
    return pd.Timestamp(timestamp_values[position])


time_boundaries = {fraction: closest_group_end(fraction) for fraction in BOUNDARY_FRACTIONS}


def interval_mask(start_fraction: float, end_fraction: float) -> np.ndarray:
    mask = np.ones(len(raw_df), dtype=bool)
    if start_fraction > 0:
        mask &= raw_df["timestamp"].gt(time_boundaries[start_fraction]).to_numpy()
    if end_fraction < 1:
        mask &= raw_df["timestamp"].le(time_boundaries[end_fraction]).to_numpy()
    return mask


final_train_mask = interval_mask(0.0, 0.70)
final_validation_mask = interval_mask(0.70, 0.80)
final_test_mask = interval_mask(0.80, 1.0)


def summarize_mask(name: str, mask: np.ndarray) -> dict:
    subset = raw_df.loc[mask]
    return {
        "split": name,
        "rows": len(subset),
        "row_ratio_pct": len(subset) / len(raw_df) * 100,
        "timestamp_groups": subset["timestamp"].nunique(),
        "positive_rows": int(subset["class"].sum()),
        "positive_rate_pct": subset["class"].mean() * 100,
        "start_time": subset["timestamp"].min(),
        "end_time": subset["timestamp"].max(),
    }


split_summary = pd.DataFrame(
    [
        summarize_mask("final_train", final_train_mask),
        summarize_mask("final_validation_calibration", final_validation_mask),
        summarize_mask("final_test", final_test_mask),
    ]
).set_index("split")

assert (final_train_mask.astype(int) + final_validation_mask.astype(int) + final_test_mask.astype(int) == 1).all()
assert split_summary["rows"].sum() == len(raw_df)
assert raw_df.loc[final_train_mask, "timestamp"].max() < raw_df.loc[final_validation_mask, "timestamp"].min()
assert raw_df.loc[final_validation_mask, "timestamp"].max() < raw_df.loc[final_test_mask, "timestamp"].min()

boundary_summary = pd.DataFrame(
    {
        "fraction": list(time_boundaries),
        "timestamp_group_end": list(time_boundaries.values()),
        "actual_cumulative_rows": [
            int(timestamp_group_sizes.loc[:boundary].sum()) for boundary in time_boundaries.values()
        ],
    }
).set_index("fraction")
boundary_summary["actual_cumulative_pct"] = boundary_summary["actual_cumulative_rows"] / len(raw_df) * 100

display(boundary_summary)
display(split_summary)
print("Final Validation/Calibration과 Final Test는 후속 gate 전까지 예측에 사용하지 않습니다.")


,timestamp_group_end,actual_cumulative_rows,actual_cumulative_pct
fraction,,,
0.3,1970-08-18 06:50:40+00:00,132082,29.999955
0.4,1970-08-21 23:32:59+00:00,176116,40.001454
0.5,1970-09-15 06:46:33+00:00,220156,50.004315
0.6,1970-09-28 05:10:37+00:00,264343,60.040566
0.7,1970-10-05 00:29:31+00:00,308189,69.999364
0.8,1970-10-13 16:54:14+00:00,352222,80.000636


,rows,row_ratio_pct,timestamp_groups,positive_rows,positive_rate_pct,start_time,end_time
split,,,,,,,
final_train,308189,69.999364,29248,1940,0.629484,1970-06-23 03:58:55+00:00,1970-10-05 00:29:31+00:00
final_validation_calibration,44033,10.001272,3401,357,0.810756,1970-10-05 00:29:59+00:00,1970-10-13 16:54:14+00:00
final_test,88052,19.999364,7093,2325,2.640485,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


Final Validation/Calibration과 Final Test는 후속 gate 전까지 예측에 사용하지 않습니다.


## 4. Mapping-aware 입력 구성

모든 Type에서 사용되지 않는 5개 검사 피처를 제거한다. 나머지 65개 피처는 각 행의 `inspection_type`과 `mapping.json`을 비교해 미사용 값을 `NaN`으로 바꾼다. 이 규칙은 외부 스키마이므로 전체 행에 동일하게 적용해도 통계적 누수가 없다.


In [4]:
CATEGORICAL_COLUMNS = [
    "inspection_type",
    "meta_feat1",
    "meta_feat2",
    "meta_feat3",
    "meta_feat4",
]
RAW_FEATURE_COLUMNS = CATEGORICAL_COLUMNS + mapped_inspection_columns


def build_mapping_aware_source(frame: pd.DataFrame) -> pd.DataFrame:
    source = frame[RAW_FEATURE_COLUMNS].copy()
    for inspection_type in sorted(frame["inspection_type"].unique()):
        valid = set(inspection_mapping[str(int(inspection_type))])
        invalid = [column for column in mapped_inspection_columns if column not in valid]
        if invalid:
            source.loc[frame["inspection_type"].eq(inspection_type), invalid] = np.nan
    return source


model_source = build_mapping_aware_source(raw_df)

mask_audit_rows = []
for inspection_type in range(5):
    type_mask = raw_df["inspection_type"].eq(inspection_type)
    valid = set(inspection_mapping[str(inspection_type)])
    invalid = [column for column in mapped_inspection_columns if column not in valid]
    invalid_non_missing = int(model_source.loc[type_mask, invalid].notna().sum().sum()) if invalid else 0
    assert invalid_non_missing == 0
    mask_audit_rows.append(
        {
            "inspection_type": inspection_type,
            "rows": int(type_mask.sum()),
            "mapped_valid_features": len(valid),
            "masked_features": len(invalid),
            "invalid_non_missing_after_mask": invalid_non_missing,
        }
    )

display(pd.DataFrame(mask_audit_rows).set_index("inspection_type"))
print("mapping-aware source shape:", model_source.shape)


,rows,mapped_valid_features,masked_features,invalid_non_missing_after_mask
inspection_type,,,,
0,97053,44,21,0
1,57673,52,13,0
2,128174,65,0,0
3,151931,65,0,0
4,5443,21,44,0


mapping-aware source shape: (440274, 70)


## 5. Train-only 전처리와 중복 전략

각 Fold/Final Train에서만 상수 검사 피처를 판정하고 One-hot encoder를 fit한다. 미래 구간은 transform만 한다. 중복은 Train에서만 `keep`, `exact_dedup`, `signature_weight` 세 전략으로 비교하며 미래 구간의 실제 빈도는 유지한다.


In [5]:
SIGNATURE_COLUMNS = [
    column for column in raw_df.columns
    if column not in {"record_id", "timestamp", "class"}
]
EXACT_DUPLICATE_COLUMNS = [column for column in raw_df.columns if column != "record_id"]


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", dtype=np.float32, sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", dtype=np.float32, sparse=False)


def fit_transformer(train_source: pd.DataFrame):
    constant_features = [
        column
        for column in mapped_inspection_columns
        if train_source[column].nunique(dropna=True) <= 1
    ]
    continuous_columns = [
        column for column in mapped_inspection_columns if column not in constant_features
    ]
    transformer = ColumnTransformer(
        transformers=[
            ("categorical", make_one_hot_encoder(), CATEGORICAL_COLUMNS),
            ("continuous", "passthrough", continuous_columns),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    train_matrix = transformer.fit_transform(train_source).astype(np.float32, copy=False)
    feature_names = transformer.get_feature_names_out().tolist()
    return transformer, train_matrix, constant_features, continuous_columns, feature_names


def transform_float32(transformer: ColumnTransformer, source: pd.DataFrame) -> np.ndarray:
    return transformer.transform(source).astype(np.float32, copy=False)


def duplicate_strategy_data(
    strategy: str,
    raw_train: pd.DataFrame,
    train_matrix: np.ndarray,
    y_train: np.ndarray,
):
    if strategy == "keep":
        return train_matrix, y_train, None, {"removed_rows": 0, "weight_sum": float(len(y_train))}

    if strategy == "exact_dedup":
        duplicate_mask = raw_train.duplicated(subset=EXACT_DUPLICATE_COLUMNS, keep="first").to_numpy()
        keep_mask = ~duplicate_mask
        return (
            train_matrix[keep_mask],
            y_train[keep_mask],
            None,
            {"removed_rows": int(duplicate_mask.sum()), "weight_sum": float(keep_mask.sum())},
        )

    if strategy == "signature_weight":
        signature = pd.util.hash_pandas_object(
            raw_train[SIGNATURE_COLUMNS], index=False, categorize=True
        )
        group_sizes = signature.map(signature.value_counts(sort=False)).to_numpy(dtype=np.float32)
        weights = np.reciprocal(group_sizes, dtype=np.float32)
        return (
            train_matrix,
            y_train,
            weights,
            {
                "removed_rows": 0,
                "weight_sum": float(weights.sum()),
                "min_weight": float(weights.min()),
                "max_weight": float(weights.max()),
            },
        )

    raise ValueError(f"지원하지 않는 중복 전략: {strategy}")


def matrix_memory_mb(*matrices: np.ndarray) -> float:
    return sum(matrix.nbytes for matrix in matrices) / 1024**2


## 6. 평가와 임계값 선택 함수

Calibration의 모든 고유 확률을 후보로 사용한다. Recall 97% 이상인 후보 중 False Call Reduction 최대, Recall 최대, threshold 최대 순서로 선택한다. 최적화 구현은 작은 합성 데이터에서 직접 구현과 일치해야 한다.


In [6]:
def evaluate_probabilities(y_true, probability, threshold) -> dict:
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=float)
    prediction = (probability >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    positives = tp + fn
    negatives = tn + fp
    return {
        "rows": int(len(y_true)),
        "positive_rows": int(positives),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": tp / positives if positives else np.nan,
        "false_call_reduction": tn / negatives if negatives else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability) if len(np.unique(y_true)) == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if positives else np.nan,
    }


def select_threshold_reference(y_true, probability, min_recall=0.97) -> dict:
    rows = []
    for threshold in np.sort(np.unique(probability))[::-1]:
        metrics = evaluate_probabilities(y_true, probability, threshold)
        if metrics["recall"] >= min_recall:
            rows.append({"threshold": float(threshold), **metrics})
    if not rows:
        raise RuntimeError(f"Recall {min_recall:.1%} 조건을 만족하는 임계값이 없습니다.")
    return max(
        rows,
        key=lambda row: (row["false_call_reduction"], row["recall"], row["threshold"]),
    )


def select_threshold(y_true, probability, min_recall=0.97) -> dict:
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=float)
    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_y = y_true[order]
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[1:] != sorted_probability[:-1], True]
    )
    cumulative_tp = np.cumsum(sorted_y, dtype=np.int64)[group_ends]
    predicted_positive = group_ends + 1
    cumulative_fp = predicted_positive - cumulative_tp
    total_positive = int(sorted_y.sum())
    total_negative = int(len(sorted_y) - total_positive)
    if total_positive == 0:
        raise RuntimeError("Calibration에 positive class가 없습니다.")
    recall = cumulative_tp / total_positive
    false_call_reduction = (
        (total_negative - cumulative_fp) / total_negative
        if total_negative
        else np.full_like(recall, np.nan, dtype=float)
    )
    eligible = np.flatnonzero(recall >= min_recall)
    if len(eligible) == 0:
        raise RuntimeError(f"Recall {min_recall:.1%} 조건을 만족하는 임계값이 없습니다.")

    eligible_false_call_reduction = false_call_reduction[eligible]
    eligible_recall = recall[eligible]
    eligible_threshold = sorted_probability[group_ends[eligible]]
    best_local_position = max(
        range(len(eligible)),
        key=lambda position: (
            eligible_false_call_reduction[position],
            eligible_recall[position],
            eligible_threshold[position],
        ),
    )
    threshold = float(eligible_threshold[best_local_position])
    return {"threshold": threshold, **evaluate_probabilities(y_true, probability, threshold)}


synthetic_y = np.array([1, 0, 1, 0, 1, 0, 0, 1], dtype=np.int8)
synthetic_probability = np.array([0.9, 0.8, 0.8, 0.4, 0.3, 0.3, 0.1, 0.05])
reference_result = select_threshold_reference(synthetic_y, synthetic_probability, min_recall=0.75)
optimized_result = select_threshold(synthetic_y, synthetic_probability, min_recall=0.75)
for key in ("threshold", "recall", "false_call_reduction", "tn", "fp", "fn", "tp"):
    assert np.isclose(reference_result[key], optimized_result[key])
display(pd.Series(optimized_result, name="threshold_function_test"))


threshold               0.300000
rows                    8.000000
positive_rows           4.000000
tn                      1.000000
fp                      3.000000
fn                      1.000000
tp                      3.000000
accuracy                0.500000
precision               0.500000
recall                  0.750000
false_call_reduction    0.250000
f1                      0.600000
roc_auc                 0.562500
pr_auc                  0.666667
Name: threshold_function_test, dtype: float64

## 7. Walk-forward 후보 학습

세 Fold에서 `duplicate_strategy × scale_pos_weight` 12개 조합, 총 36개 모델을 학습한다. Fold Calibration은 Early Stopping과 임계값 선택에만 사용하고, Evaluation은 해당 Fold의 학습이나 임계값에 절대 사용하지 않는다.


In [7]:
WALK_FORWARD_FOLDS = [
    {"fold": 1, "train": (0.00, 0.30), "calibration": (0.30, 0.40), "evaluation": (0.40, 0.50)},
    {"fold": 2, "train": (0.00, 0.40), "calibration": (0.40, 0.50), "evaluation": (0.50, 0.60)},
    {"fold": 3, "train": (0.00, 0.50), "calibration": (0.50, 0.60), "evaluation": (0.60, 0.70)},
]
DUPLICATE_STRATEGIES = ("keep", "exact_dedup", "signature_weight")
SCALE_POS_WEIGHT_RULES = ("1", "25", "50", "R")

XGB_COMMON_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 3000,
    "learning_rate": 0.03,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0.0,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "early_stopping_rounds": 100,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}


def resolve_scale_pos_weight(rule: str, y_train: np.ndarray) -> float:
    if rule == "R":
        positive = int(np.sum(y_train == 1))
        negative = int(np.sum(y_train == 0))
        if positive == 0:
            raise RuntimeError("Train에 positive class가 없습니다.")
        return negative / positive
    return float(rule)


def make_xgb_classifier(scale_pos_weight: float) -> XGBClassifier:
    return XGBClassifier(scale_pos_weight=scale_pos_weight, **XGB_COMMON_PARAMS)


walk_forward_rows = []
walk_forward_started = time.perf_counter()

for fold_spec in WALK_FORWARD_FOLDS:
    fold_number = fold_spec["fold"]
    train_mask = interval_mask(*fold_spec["train"])
    calibration_mask = interval_mask(*fold_spec["calibration"])
    evaluation_mask = interval_mask(*fold_spec["evaluation"])
    assert (train_mask.astype(int) + calibration_mask.astype(int) + evaluation_mask.astype(int) <= 1).all()
    assert raw_df.loc[train_mask, "timestamp"].max() < raw_df.loc[calibration_mask, "timestamp"].min()
    assert raw_df.loc[calibration_mask, "timestamp"].max() < raw_df.loc[evaluation_mask, "timestamp"].min()

    train_source = model_source.loc[train_mask]
    calibration_source = model_source.loc[calibration_mask]
    evaluation_source = model_source.loc[evaluation_mask]
    raw_train = raw_df.loc[train_mask]
    y_train_all = raw_train["class"].to_numpy(dtype=np.int8)
    y_calibration = raw_df.loc[calibration_mask, "class"].to_numpy(dtype=np.int8)
    y_evaluation = raw_df.loc[evaluation_mask, "class"].to_numpy(dtype=np.int8)

    transformer, X_train_all, constant_features, continuous_columns, feature_names = fit_transformer(train_source)
    X_calibration = transform_float32(transformer, calibration_source)
    X_evaluation = transform_float32(transformer, evaluation_source)

    print(
        f"Fold {fold_number}: train={len(y_train_all):,}, calibration={len(y_calibration):,}, "
        f"evaluation={len(y_evaluation):,}, features={len(feature_names)}, "
        f"matrix_memory={matrix_memory_mb(X_train_all, X_calibration, X_evaluation):,.1f} MB"
    )

    for duplicate_strategy in DUPLICATE_STRATEGIES:
        X_train, y_train, sample_weight, duplicate_audit = duplicate_strategy_data(
            duplicate_strategy, raw_train, X_train_all, y_train_all
        )
        for scale_rule in SCALE_POS_WEIGHT_RULES:
            scale_value = resolve_scale_pos_weight(scale_rule, y_train)
            model = make_xgb_classifier(scale_value)
            run_started = time.perf_counter()
            fit_kwargs = {
                "X": X_train,
                "y": y_train,
                "eval_set": [(X_calibration, y_calibration)],
                "verbose": False,
            }
            if sample_weight is not None:
                fit_kwargs["sample_weight"] = sample_weight
            model.fit(**fit_kwargs)

            calibration_probability = model.predict_proba(X_calibration)[:, 1]
            threshold_result = select_threshold(
                y_calibration,
                calibration_probability,
                min_recall=CALIBRATION_MIN_RECALL,
            )
            evaluation_probability = model.predict_proba(X_evaluation)[:, 1]
            evaluation_metrics = evaluate_probabilities(
                y_evaluation, evaluation_probability, threshold_result["threshold"]
            )

            row = {
                "fold": fold_number,
                "duplicate_strategy": duplicate_strategy,
                "scale_rule": scale_rule,
                "scale_value": float(scale_value),
                "train_rows": int(len(y_train)),
                "train_positive_rows": int(y_train.sum()),
                "removed_train_rows": int(duplicate_audit["removed_rows"]),
                "train_weight_sum": float(duplicate_audit["weight_sum"]),
                "constant_feature_count": len(constant_features),
                "encoded_feature_count": len(feature_names),
                "best_iteration": int(model.best_iteration),
                "best_calibration_aucpr": float(model.best_score),
                "threshold": float(threshold_result["threshold"]),
                "calibration_pr_auc": float(threshold_result["pr_auc"]),
                "calibration_recall": float(threshold_result["recall"]),
                "calibration_false_call_reduction": float(threshold_result["false_call_reduction"]),
                "calibration_tp": int(threshold_result["tp"]),
                "calibration_fn": int(threshold_result["fn"]),
                "calibration_fp": int(threshold_result["fp"]),
                "calibration_tn": int(threshold_result["tn"]),
                "evaluation_pr_auc": float(evaluation_metrics["pr_auc"]),
                "evaluation_recall": float(evaluation_metrics["recall"]),
                "evaluation_false_call_reduction": float(evaluation_metrics["false_call_reduction"]),
                "evaluation_tp": int(evaluation_metrics["tp"]),
                "evaluation_fn": int(evaluation_metrics["fn"]),
                "evaluation_fp": int(evaluation_metrics["fp"]),
                "evaluation_tn": int(evaluation_metrics["tn"]),
                "evaluation_recall_97": bool(evaluation_metrics["recall"] >= CALIBRATION_MIN_RECALL),
                "run_seconds": time.perf_counter() - run_started,
            }
            walk_forward_rows.append(row)
            print(
                f"  {duplicate_strategy:16s} spw={scale_rule:>2s} "
                f"best={model.best_iteration:4d} threshold={row['threshold']:.6f} "
                f"eval_pr_auc={row['evaluation_pr_auc']:.4f} "
                f"eval_recall={row['evaluation_recall']:.4f} "
                f"eval_fcr={row['evaluation_false_call_reduction']:.4f}"
            )
            del model, calibration_probability, evaluation_probability
            gc.collect()

    del transformer, X_train_all, X_calibration, X_evaluation
    gc.collect()

walk_forward_results = pd.DataFrame(walk_forward_rows)
print(f"walk-forward total minutes: {(time.perf_counter() - walk_forward_started) / 60:.2f}")
display(walk_forward_results)


Fold 1: train=132,082, calibration=44,034, evaluation=44,040, features=176, matrix_memory=147.8 MB
  keep             spw= 1 best= 146 threshold=0.002079 eval_pr_auc=0.1807 eval_recall=0.9847 eval_fcr=0.1791
  keep             spw=25 best= 224 threshold=0.026175 eval_pr_auc=0.2439 eval_recall=0.9877 eval_fcr=0.2751
  keep             spw=50 best= 185 threshold=0.050884 eval_pr_auc=0.2486 eval_recall=0.9877 eval_fcr=0.2736
  keep             spw= R best= 226 threshold=0.063170 eval_pr_auc=0.2774 eval_recall=1.0000 eval_fcr=0.1322
  exact_dedup      spw= 1 best= 137 threshold=0.002269 eval_pr_auc=0.1778 eval_recall=0.9387 eval_fcr=0.2054
  exact_dedup      spw=25 best= 117 threshold=0.043440 eval_pr_auc=0.2344 eval_recall=0.9847 eval_fcr=0.2922
  exact_dedup      spw=50 best= 117 threshold=0.069787 eval_pr_auc=0.2002 eval_recall=0.9847 eval_fcr=0.2099
  exact_dedup      spw= R best=   9 threshold=0.425557 eval_pr_auc=0.1714 eval_recall=1.0000 eval_fcr=0.0000
  signature_weight spw= 1 bes

,fold,duplicate_strategy,scale_rule,scale_value,train_rows,train_positive_rows,removed_train_rows,train_weight_sum,constant_feature_count,encoded_feature_count,...,calibration_tn,evaluation_pr_auc,evaluation_recall,evaluation_false_call_reduction,evaluation_tp,evaluation_fn,evaluation_fp,evaluation_tn,evaluation_recall_97,run_seconds
0,1,keep,1,1.000000,132082,1223,0,132082.000000,10,176,...,5598,0.180693,0.984663,0.179119,321,5,35884,7830,True,9.058257
1,1,keep,25,25.000000,132082,1223,0,132082.000000,10,176,...,18633,0.243876,0.987730,0.275106,322,4,31688,12026,True,12.078346
2,1,keep,50,50.000000,132082,1223,0,132082.000000,10,176,...,17652,0.248574,0.987730,0.273619,322,4,31753,11961,True,10.998529
3,1,keep,R,106.998365,132082,1223,0,132082.000000,10,176,...,13772,0.277413,1.000000,0.132246,326,0,37933,5781,True,11.927051
4,1,exact_dedup,1,1.000000,130316,1203,1766,130316.000000,10,176,...,10098,0.177820,0.938650,0.205449,306,20,34733,8981,False,8.684719
5,1,exact_dedup,25,25.000000,130316,1203,1766,130316.000000,10,176,...,12634,0.234423,0.984663,0.292195,321,5,30941,12773,True,7.678213
6,1,exact_dedup,50,50.000000,130316,1203,1766,130316.000000,10,176,...,7950,0.200233,0.984663,0.209933,321,5,34537,9177,True,7.687393
7,1,exact_dedup,R,107.325852,130316,1203,1766,130316.000000,10,176,...,0,0.171411,1.000000,0.000000,326,0,43714,0,True,3.909185
8,1,signature_weight,1,1.000000,132082,1223,0,119241.000000,10,176,...,9970,0.175367,0.938650,0.182321,306,20,35744,7970,False,7.769390
9,1,signature_weight,25,25.000000,132082,1223,0,119241.000000,10,176,...,7879,0.224316,0.984663,0.175619,321,5,36037,7677,True,8.709869


## 8. Walk-forward 후보 선택

세 Evaluation Recall이 모두 95% 이상인 후보만 통과한다. 통과 후보 중 평균 PR-AUC 최대, 평균 False Call Reduction 최대, 최저 Recall 최대, 단순 전략, 작은 가중치 순서로 하나를 선택한다. 후보가 없으면 Final Validation과 Final Test를 실행하지 않는다.


In [8]:
strategy_rank = {"keep": 0, "exact_dedup": 1, "signature_weight": 2}
scale_rank = {"1": 0, "25": 1, "50": 2, "R": 3}

candidate_summary = (
    walk_forward_results
    .groupby(["duplicate_strategy", "scale_rule"], as_index=False)
    .agg(
        folds=("fold", "nunique"),
        all_evaluation_recall_95=("evaluation_recall", lambda values: bool((values >= EVALUATION_MIN_RECALL).all())),
        evaluation_recall_97_folds=("evaluation_recall_97", "sum"),
        mean_evaluation_pr_auc=("evaluation_pr_auc", "mean"),
        mean_evaluation_false_call_reduction=("evaluation_false_call_reduction", "mean"),
        min_evaluation_recall=("evaluation_recall", "min"),
        mean_evaluation_recall=("evaluation_recall", "mean"),
        mean_best_iteration=("best_iteration", "mean"),
        total_run_seconds=("run_seconds", "sum"),
    )
)
candidate_summary["strategy_rank"] = candidate_summary["duplicate_strategy"].map(strategy_rank)
candidate_summary["scale_rank"] = candidate_summary["scale_rule"].map(scale_rank)

eligible_candidates = candidate_summary.loc[candidate_summary["all_evaluation_recall_95"]].copy()
eligible_candidates = eligible_candidates.sort_values(
    [
        "mean_evaluation_pr_auc",
        "mean_evaluation_false_call_reduction",
        "min_evaluation_recall",
        "strategy_rank",
        "scale_rank",
    ],
    ascending=[False, False, False, True, True],
    kind="stable",
)

EXPERIMENT_GATE_PASSED = not eligible_candidates.empty
selected_config = eligible_candidates.iloc[0].to_dict() if EXPERIMENT_GATE_PASSED else None

display(candidate_summary.sort_values("mean_evaluation_pr_auc", ascending=False))
print("walk-forward safety gate:", "PASS" if EXPERIMENT_GATE_PASSED else "FAIL")
if selected_config is not None:
    display(pd.Series(selected_config, name="selected_configuration"))
else:
    print("모든 후보가 Evaluation Recall 95% gate를 통과하지 못했습니다.")
    print("명세에 따라 Final Validation/Calibration과 Final Test를 실행하지 않습니다.")


,duplicate_strategy,scale_rule,folds,all_evaluation_recall_95,evaluation_recall_97_folds,mean_evaluation_pr_auc,mean_evaluation_false_call_reduction,min_evaluation_recall,mean_evaluation_recall,mean_best_iteration,total_run_seconds,strategy_rank,scale_rank
7,keep,R,3,False,2,0.130581,0.176793,0.947368,0.973909,109.333333,23.875687,0,3
5,keep,25,3,True,2,0.124168,0.183629,0.960526,0.974205,122.666667,27.729390,0,1
6,keep,50,3,False,1,0.122476,0.295363,0.948718,0.967851,104.000000,26.293995,0,2
1,exact_dedup,25,3,False,2,0.111765,0.238254,0.947368,0.977344,88.666667,21.860400,1,1
9,signature_weight,25,3,False,2,0.109003,0.135926,0.947368,0.977344,83.666667,21.872511,2,1
11,signature_weight,R,3,True,3,0.106920,0.212538,0.974359,0.980784,87.666667,21.656794,2,3
10,signature_weight,50,3,False,2,0.103701,0.220443,0.947368,0.978366,99.000000,23.628626,2,2
8,signature_weight,1,3,False,1,0.101741,0.152314,0.938650,0.964199,83.000000,20.578811,2,0
2,exact_dedup,50,3,False,2,0.096064,0.206094,0.934211,0.972958,86.666667,23.811424,1,2
0,exact_dedup,1,3,False,1,0.093004,0.208949,0.938650,0.964199,91.666667,22.420404,1,0


walk-forward safety gate: PASS


duplicate_strategy                            keep
scale_rule                                      25
folds                                            3
all_evaluation_recall_95                      True
evaluation_recall_97_folds                       2
mean_evaluation_pr_auc                    0.124168
mean_evaluation_false_call_reduction      0.183629
min_evaluation_recall                     0.960526
mean_evaluation_recall                    0.974205
mean_best_iteration                     122.666667
total_run_seconds                         27.72939
strategy_rank                                    0
scale_rank                                       1
Name: selected_configuration, dtype: object

## 9. 최종 모델 학습과 Final Validation/Calibration

Gate를 통과한 경우에만 앞 70% 전체로 최종 모델을 학습한다. 트리 분기와 가중치는 Final Train으로만 학습하며, 70~80% 라벨은 gradient 업데이트가 아니라 `best_iteration`과 최종 Recall 97% 임계값 결정에만 사용한다.


In [9]:
final_model = None
final_transformer = None
final_threshold_result = None
final_feature_names = None
final_constant_features = None
final_scale_value = None
pretest_artifact = None

if EXPERIMENT_GATE_PASSED:
    final_train_source = model_source.loc[final_train_mask]
    final_validation_source = model_source.loc[final_validation_mask]
    final_raw_train = raw_df.loc[final_train_mask]
    y_final_train_all = final_raw_train["class"].to_numpy(dtype=np.int8)
    y_final_validation = raw_df.loc[final_validation_mask, "class"].to_numpy(dtype=np.int8)

    (
        final_transformer,
        X_final_train_all,
        final_constant_features,
        final_continuous_columns,
        final_feature_names,
    ) = fit_transformer(final_train_source)
    X_final_validation = transform_float32(final_transformer, final_validation_source)

    selected_duplicate_strategy = selected_config["duplicate_strategy"]
    selected_scale_rule = selected_config["scale_rule"]
    X_final_train, y_final_train, final_sample_weight, final_duplicate_audit = duplicate_strategy_data(
        selected_duplicate_strategy,
        final_raw_train,
        X_final_train_all,
        y_final_train_all,
    )
    final_scale_value = resolve_scale_pos_weight(selected_scale_rule, y_final_train)
    final_model = make_xgb_classifier(final_scale_value)
    final_fit_kwargs = {
        "X": X_final_train,
        "y": y_final_train,
        "eval_set": [(X_final_validation, y_final_validation)],
        "verbose": False,
    }
    if final_sample_weight is not None:
        final_fit_kwargs["sample_weight"] = final_sample_weight
    final_model.fit(**final_fit_kwargs)

    final_validation_probability = final_model.predict_proba(X_final_validation)[:, 1]
    final_threshold_result = select_threshold(
        y_final_validation,
        final_validation_probability,
        min_recall=CALIBRATION_MIN_RECALL,
    )

    final_selection_summary = pd.Series(
        {
            "duplicate_strategy": selected_duplicate_strategy,
            "scale_pos_weight_rule": selected_scale_rule,
            "scale_pos_weight_value": final_scale_value,
            "final_train_rows": len(y_final_train),
            "final_train_positive_rows": int(y_final_train.sum()),
            "removed_train_rows": final_duplicate_audit["removed_rows"],
            "train_weight_sum": final_duplicate_audit["weight_sum"],
            "constant_features_removed": len(final_constant_features),
            "encoded_features": len(final_feature_names),
            "best_iteration": int(final_model.best_iteration),
            "best_validation_aucpr": float(final_model.best_score),
            "decision_threshold": final_threshold_result["threshold"],
            "validation_pr_auc": final_threshold_result["pr_auc"],
            "validation_recall": final_threshold_result["recall"],
            "validation_false_call_reduction": final_threshold_result["false_call_reduction"],
            "validation_tp": final_threshold_result["tp"],
            "validation_fn": final_threshold_result["fn"],
            "validation_fp": final_threshold_result["fp"],
            "validation_tn": final_threshold_result["tn"],
        },
        name="locked_final_configuration",
    )
    assert final_threshold_result["recall"] >= CALIBRATION_MIN_RECALL
    display(final_selection_summary)
else:
    print("Walk-forward gate FAIL: 최종 모델 학습과 Final Validation/Calibration을 건너뜁니다.")


duplicate_strategy                     keep
scale_pos_weight_rule                    25
scale_pos_weight_value                 25.0
final_train_rows                     308189
final_train_positive_rows              1940
removed_train_rows                        0
train_weight_sum                   308189.0
constant_features_removed                 6
encoded_features                        191
best_iteration                          123
best_validation_aucpr              0.324212
decision_threshold                 0.050806
validation_pr_auc                  0.313306
validation_recall                  0.971989
validation_false_call_reduction    0.618417
validation_tp                           347
validation_fn                            10
validation_fp                         16666
validation_tn                         27010
Name: locked_final_configuration, dtype: object

## 10. 모델 artifact 저장과 reload 검증

Final Test 전에 모델·전처리·피처·매핑·기간·가중치·고정 임계값을 하나의 artifact로 저장한다. 저장 전후 Final Validation 확률과 판정이 정확히 일치해야 한다.


In [10]:
if EXPERIMENT_GATE_PASSED:
    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    pretest_artifact = {
        "experiment_id": EXPERIMENT_ID,
        "model": final_model,
        "preprocessor": final_transformer,
        "mapping": inspection_mapping,
        "mapping_sha256": MAPPING_SHA256,
        "data_sha256": DATA_SHA256,
        "raw_feature_columns": RAW_FEATURE_COLUMNS,
        "final_feature_names": final_feature_names,
        "globally_unmapped_features": globally_unmapped_features,
        "train_constant_features": final_constant_features,
        "duplicate_strategy": selected_config["duplicate_strategy"],
        "scale_pos_weight_rule": selected_config["scale_rule"],
        "scale_pos_weight_value": float(final_scale_value),
        "decision_threshold": float(final_threshold_result["threshold"]),
        "threshold_selection_rule": "Recall >= 0.97; maximize FCR, then Recall, then threshold",
        "random_state": RANDOM_STATE,
        "train_start_time": raw_df.loc[final_train_mask, "timestamp"].min().isoformat(),
        "train_end_time": raw_df.loc[final_train_mask, "timestamp"].max().isoformat(),
        "validation_start_time": raw_df.loc[final_validation_mask, "timestamp"].min().isoformat(),
        "validation_end_time": raw_df.loc[final_validation_mask, "timestamp"].max().isoformat(),
        "test_start_time": raw_df.loc[final_test_mask, "timestamp"].min().isoformat(),
        "test_end_time": raw_df.loc[final_test_mask, "timestamp"].max().isoformat(),
        "best_iteration": int(final_model.best_iteration),
        "library_versions": LIBRARY_VERSIONS,
        "walk_forward_summary": candidate_summary.drop(columns=["strategy_rank", "scale_rank"]).to_dict("records"),
    }
    joblib.dump(pretest_artifact, MODEL_PATH)
    loaded_artifact = joblib.load(MODEL_PATH)

    reload_rows = min(1024, int(final_validation_mask.sum()))
    reload_source = model_source.loc[final_validation_mask].iloc[:reload_rows]
    original_matrix = final_transformer.transform(reload_source).astype(np.float32, copy=False)
    loaded_matrix = loaded_artifact["preprocessor"].transform(reload_source).astype(np.float32, copy=False)
    original_probability = final_model.predict_proba(original_matrix)[:, 1]
    loaded_probability = loaded_artifact["model"].predict_proba(loaded_matrix)[:, 1]
    assert np.array_equal(original_matrix, loaded_matrix, equal_nan=True)
    assert np.allclose(original_probability, loaded_probability, rtol=0, atol=0)
    assert np.array_equal(
        original_probability >= final_threshold_result["threshold"],
        loaded_probability >= loaded_artifact["decision_threshold"],
    )
    print("saved:", MODEL_PATH)
    print("artifact reload prediction check: PASS")
else:
    print("Walk-forward gate FAIL: 모델 artifact를 저장하지 않습니다.")


saved: /Users/peace/Downloads/2026 여름방학 AI 제조 부캠/최종프로젝트/raw_dataset/siemens_aoi_ML_practice/models/0824_peace_002_mapping_aware_xgboost.pkl
artifact reload prediction check: PASS


## 11. Final Test 실행 게이트

아래 조건이 모두 참일 때만 마지막 20%를 변환·추론한다. Test 코드에는 임계값을 계산하는 함수 호출이 없으며 저장된 고정 임계값만 사용한다.


In [11]:
test_gate = {
    "walk_forward_gate_passed": bool(EXPERIMENT_GATE_PASSED),
    "final_model_fitted": final_model is not None,
    "final_preprocessor_fitted": final_transformer is not None,
    "threshold_locked": final_threshold_result is not None,
    "artifact_saved": MODEL_PATH.exists() if EXPERIMENT_GATE_PASSED else False,
    "validation_recall_97": (
        bool(final_threshold_result["recall"] >= CALIBRATION_MIN_RECALL)
        if final_threshold_result is not None
        else False
    ),
}
TEST_GATE_PASSED = all(test_gate.values())
display(pd.Series(test_gate, name="final_test_gate"))
print("Final Test gate:", "PASS" if TEST_GATE_PASSED else "BLOCKED")


walk_forward_gate_passed     True
final_model_fitted           True
final_preprocessor_fitted    True
threshold_locked             True
artifact_saved               True
validation_recall_97         True
Name: final_test_gate, dtype: bool

Final Test gate: PASS


## 12. Final Test 1회 평가

저장된 모델과 임계값을 변경하지 않고 최종 Test를 한 번 평가한다. Test Recall 95%는 합격 기준이지만, 미달하더라도 이 Test에서 임계값을 다시 조정하지 않는다.


In [12]:
test_probability = None
test_prediction = None
test_metrics = None

if TEST_GATE_PASSED:
    locked_threshold = float(pretest_artifact["decision_threshold"])
    assert locked_threshold == float(final_threshold_result["threshold"])

    final_test_source = model_source.loc[final_test_mask]
    X_final_test = final_transformer.transform(final_test_source).astype(np.float32, copy=False)
    y_final_test = raw_df.loc[final_test_mask, "class"].to_numpy(dtype=np.int8)
    test_probability = final_model.predict_proba(X_final_test)[:, 1]
    test_prediction = (test_probability >= locked_threshold).astype(np.int8)
    test_metrics = evaluate_probabilities(y_final_test, test_probability, locked_threshold)
    test_metrics["threshold"] = locked_threshold
    test_metrics["test_recall_95_passed"] = bool(test_metrics["recall"] >= EVALUATION_MIN_RECALL)
    test_metrics["pr_auc_improved_over_baseline"] = bool(test_metrics["pr_auc"] > BASELINE_TEST_PR_AUC)

    final_test_table = pd.DataFrame(
        [
            {
                "Model": "Mapping-aware integrated XGBoost",
                "PR-AUC": test_metrics["pr_auc"],
                "Threshold": locked_threshold,
                "Real Defect Recall": test_metrics["recall"],
                "TP": test_metrics["tp"],
                "FN": test_metrics["fn"],
                "FP": test_metrics["fp"],
                "TN": test_metrics["tn"],
                "False Call Reduction": test_metrics["false_call_reduction"],
            }
        ]
    )
    display(final_test_table)
    print("Test Recall 95%:", "PASS" if test_metrics["test_recall_95_passed"] else "FAIL")
    print("PR-AUC > baseline:", test_metrics["pr_auc_improved_over_baseline"])
else:
    print("Final Test gate가 닫혀 있어 Test를 읽거나 추론하지 않았습니다.")


,Model,PR-AUC,Threshold,Real Defect Recall,TP,FN,FP,TN,False Call Reduction
0,Mapping-aware integrated XGBoost,0.186329,0.050806,0.850753,1978,347,35669,50058,0.583923


Test Recall 95%: FAIL
PR-AUC > baseline: False


## 13. 하위그룹 평가

Test가 실행된 경우 Type, 주차, seen/unseen input signature, 전체 기간 label-conflict signature별 성능을 같은 고정 임계값으로 계산한다. 하위그룹에 positive가 없으면 PR-AUC와 Recall은 `NaN`으로 유지한다.


In [13]:
def subgroup_evaluation(frame, probability, threshold, group_values, group_name):
    rows = []
    group_array = np.asarray(group_values)
    y_array = frame["class"].to_numpy(dtype=np.int8)
    for group_value in pd.unique(group_array):
        mask = group_array == group_value
        metrics = evaluate_probabilities(y_array[mask], probability[mask], threshold)
        rows.append({group_name: group_value, **metrics})
    return pd.DataFrame(rows)


if TEST_GATE_PASSED:
    raw_test = raw_df.loc[final_test_mask].copy()
    raw_history = raw_df.loc[~final_test_mask]
    locked_threshold = float(pretest_artifact["decision_threshold"])

    type_metrics = subgroup_evaluation(
        raw_test,
        test_probability,
        locked_threshold,
        raw_test["inspection_type"].to_numpy(),
        "inspection_type",
    )
    week_labels = raw_test["timestamp"].dt.strftime("%Y-W%W").to_numpy()
    weekly_metrics = subgroup_evaluation(
        raw_test, test_probability, locked_threshold, week_labels, "week"
    )

    history_signature = pd.util.hash_pandas_object(
        raw_history[SIGNATURE_COLUMNS], index=False, categorize=True
    )
    test_signature = pd.util.hash_pandas_object(
        raw_test[SIGNATURE_COLUMNS], index=False, categorize=True
    )
    history_signature_set = set(history_signature.to_numpy())
    seen_labels = np.where(test_signature.isin(history_signature_set), "seen", "unseen")
    seen_metrics = subgroup_evaluation(
        raw_test, test_probability, locked_threshold, seen_labels, "signature_status"
    )

    all_signature = pd.util.hash_pandas_object(
        raw_df[SIGNATURE_COLUMNS], index=False, categorize=True
    )
    signature_label_counts = (
        pd.DataFrame({"signature": all_signature.to_numpy(), "class": raw_df["class"].to_numpy()})
        .groupby("signature", sort=False)["class"]
        .nunique()
    )
    conflict_signatures = set(signature_label_counts.index[signature_label_counts > 1].to_numpy())
    conflict_labels = np.where(test_signature.isin(conflict_signatures), "label_conflict", "no_conflict")
    conflict_metrics = subgroup_evaluation(
        raw_test, test_probability, locked_threshold, conflict_labels, "conflict_status"
    )

    print("inspection type metrics")
    display(type_metrics)
    print("weekly metrics")
    display(weekly_metrics)
    print("seen/unseen signature metrics")
    display(seen_metrics)
    print("label-conflict signature metrics (post-test analysis)")
    display(conflict_metrics)
else:
    type_metrics = weekly_metrics = seen_metrics = conflict_metrics = pd.DataFrame()
    print("Final Test가 실행되지 않아 하위그룹 평가를 건너뜁니다.")


inspection type metrics


,inspection_type,rows,positive_rows,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
0,2,20543,731,7770,12042,97,634,0.409093,0.050016,0.867305,0.392187,0.094577,0.830292,0.476851
1,3,34939,612,25942,8385,131,481,0.756261,0.054252,0.785948,0.755732,0.101498,0.850060,0.255530
2,0,19491,195,15495,3801,112,83,0.799241,0.021370,0.425641,0.803016,0.040696,0.721537,0.074236
3,1,12351,774,851,10726,7,767,0.131002,0.066736,0.990956,0.073508,0.125051,0.827997,0.209135
4,4,728,13,0,715,0,13,0.017857,0.017857,1.000000,0.000000,0.035088,0.903658,0.399320


weekly metrics


,week,rows,positive_rows,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
0,1970-W41,18085,169,8286,9630,30,139,0.465856,0.014229,0.822485,0.462492,0.027973,0.836553,0.417047
1,1970-W42,32314,787,20480,11047,96,691,0.655165,0.058869,0.878018,0.649602,0.110339,0.879499,0.174138
2,1970-W43,36445,1364,20858,14223,221,1143,0.603677,0.074385,0.837977,0.594567,0.136641,0.813499,0.198549
3,1970-W44,1208,5,434,769,0,5,0.363411,0.006460,1.000000,0.360765,0.012837,0.938321,0.553610


seen/unseen signature metrics


,signature_status,rows,positive_rows,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
0,unseen,79537,2226,45987,31324,338,1888,0.601921,0.056847,0.848158,0.594831,0.106552,0.848452,0.190334
1,seen,8515,99,4071,4345,9,90,0.488667,0.020293,0.909091,0.483721,0.039700,0.829433,0.118724


label-conflict signature metrics (post-test analysis)


,conflict_status,rows,positive_rows,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
0,no_conflict,85433,2215,50033,33185,336,1879,0.607634,0.053588,0.848307,0.601228,0.100807,0.851782,0.189583
1,label_conflict,2619,110,25,2484,11,99,0.047346,0.038328,0.900000,0.009964,0.073524,0.694081,0.240632


## 14. False Negative와 피처 중요도 분석

False Negative가 특정 검사유형·주차·메타 범주에 집중되는지 집계한다. XGBoost gain importance는 연관성 설명에만 사용하며 인과관계로 해석하지 않는다.


In [14]:
if TEST_GATE_PASSED:
    analysis_frame = raw_test[
        ["inspection_type", "meta_feat1", "meta_feat2", "meta_feat3", "meta_feat4", "timestamp", "class"]
    ].copy()
    analysis_frame["prediction"] = test_prediction
    analysis_frame["week"] = analysis_frame["timestamp"].dt.strftime("%Y-W%W")
    false_negative_frame = analysis_frame.loc[
        analysis_frame["class"].eq(1) & analysis_frame["prediction"].eq(0)
    ]

    fn_by_type = false_negative_frame.groupby("inspection_type").size().rename("fn_rows").to_frame()
    fn_by_week = false_negative_frame.groupby("week").size().rename("fn_rows").to_frame()
    fn_by_meta = {
        column: false_negative_frame.groupby(column).size().sort_values(ascending=False).head(10).rename("fn_rows").to_frame()
        for column in ["meta_feat1", "meta_feat2", "meta_feat3", "meta_feat4"]
    }

    gain_scores = final_model.get_booster().get_score(importance_type="gain")
    feature_importance = (
        pd.DataFrame(
            {
                "feature": final_feature_names,
                "gain": [gain_scores.get(f"f{position}", 0.0) for position in range(len(final_feature_names))],
            }
        )
        .sort_values("gain", ascending=False)
        .head(30)
        .reset_index(drop=True)
    )

    print("False Negative by inspection_type")
    display(fn_by_type)
    print("False Negative by week")
    display(fn_by_week)
    for column, table in fn_by_meta.items():
        print(f"False Negative top categories: {column}")
        display(table)
    print("Top 30 feature gain importance (not causal)")
    display(feature_importance)
else:
    print("Final Test가 실행되지 않아 오류 분석과 최종 피처 중요도를 건너뜁니다.")


False Negative by inspection_type


,fn_rows
inspection_type,
0,112
1,7
2,97
3,131


False Negative by week


,fn_rows
week,
1970-W41,30
1970-W42,96
1970-W43,221


False Negative top categories: meta_feat1


...

,fn_rows
meta_feat3,
1,228
0,119


False Negative top categories: meta_feat4


,fn_rows
meta_feat4,
2,110
5,26
6,24
23,21
8,19
7,17
13,16
12,14
26,12


Top 30 feature gain importance (not causal)


,feature,gain
0,continuous__inspection_feat95,3324.679199
1,categorical__meta_feat1_27,3022.312500
2,continuous__inspection_feat48,2562.124756
3,categorical__meta_feat1_15,2412.739258
4,categorical__inspection_type_3,1898.694824
5,categorical__meta_feat1_16,1626.822388
6,categorical__meta_feat1_3,1605.363037
7,categorical__meta_feat1_35,1394.287354
8,categorical__meta_feat4_37,1108.415527
9,continuous__inspection_feat25,1100.029663


## 15. 원본 무결성, 최종 artifact와 결론

원본 SHA-256가 실행 전후 같은지 확인한다. Test가 실행됐다면 성능을 artifact 메타데이터에만 추가하고 모델 가중치·전처리·임계값과 reload 예측이 바뀌지 않았음을 다시 검증한다.


In [15]:
assert sha256_file(DATA_PATH) == DATA_SHA256
assert sha256_file(MAPPING_PATH) == MAPPING_SHA256

if TEST_GATE_PASSED:
    booster_hash_before = hashlib.sha256(final_model.get_booster().save_raw()).hexdigest()
    pretest_artifact["test_evaluation"] = test_metrics
    pretest_artifact["test_subgroup_summary"] = {
        "inspection_type": type_metrics.to_dict("records"),
        "weekly": weekly_metrics.to_dict("records"),
        "seen_unseen": seen_metrics.to_dict("records"),
        "label_conflict": conflict_metrics.to_dict("records"),
    }
    joblib.dump(pretest_artifact, MODEL_PATH)
    final_loaded_artifact = joblib.load(MODEL_PATH)
    booster_hash_after = hashlib.sha256(
        final_loaded_artifact["model"].get_booster().save_raw()
    ).hexdigest()
    assert booster_hash_before == booster_hash_after
    assert final_loaded_artifact["decision_threshold"] == final_threshold_result["threshold"]

    verification_matrix = final_loaded_artifact["preprocessor"].transform(
        model_source.loc[final_validation_mask].iloc[:1024]
    ).astype(np.float32, copy=False)
    verification_probability = final_loaded_artifact["model"].predict_proba(verification_matrix)[:, 1]
    assert np.allclose(verification_probability, loaded_probability, rtol=0, atol=0)

    experiment_status = "PASS" if test_metrics["recall"] >= EVALUATION_MIN_RECALL else "TARGET_NOT_MET"
    final_summary = pd.Series(
        {
            "experiment_status": experiment_status,
            "walk_forward_gate": "PASS",
            "test_recall_95": test_metrics["test_recall_95_passed"],
            "test_pr_auc": test_metrics["pr_auc"],
            "test_threshold": test_metrics["threshold"],
            "test_recall": test_metrics["recall"],
            "test_false_call_reduction": test_metrics["false_call_reduction"],
            "test_tp": test_metrics["tp"],
            "test_fn": test_metrics["fn"],
            "test_fp": test_metrics["fp"],
            "model_path": str(MODEL_PATH),
            "source_integrity": "PASS",
            "artifact_reload": "PASS",
        },
        name="final_experiment_summary",
    )
    display(final_summary)
    if experiment_status == "TARGET_NOT_MET":
        print("Test Recall 95% 미달입니다. Test 결과로 임계값을 조정하지 않습니다.")
else:
    final_summary = pd.Series(
        {
            "experiment_status": "WALK_FORWARD_GATE_FAILED",
            "walk_forward_gate": "FAIL",
            "final_validation_used": False,
            "final_test_used": False,
            "model_saved": False,
            "source_integrity": "PASS",
        },
        name="final_experiment_summary",
    )
    display(final_summary)
    print("후속 실험에서 피처, 불균형 보정, 파라미터 또는 데이터·라벨 품질을 개선합니다.")


experiment_status                                               TARGET_NOT_MET
walk_forward_gate                                                         PASS
test_recall_95                                                           False
test_pr_auc                                                           0.186329
test_threshold                                                        0.050806
test_recall                                                           0.850753
test_false_call_reduction                                             0.583923
test_tp                                                                   1978
test_fn                                                                    347
test_fp                                                                  35669
model_path                   /Users/peace/Downloads/2026 여름방학 AI 제ᄌ...
source_integrity                                                          PASS
artifact_reload                                     

Test Recall 95% 미달입니다. Test 결과로 임계값을 조정하지 않습니다.


## 16. 실행 후 문서 반영

전체 실행이 끝나면 실제 출력값만 사용해 다음을 갱신한다.

1. `docs/experiments/0824_peace_002_mapping_aware_xgboost.md`의 상태와 결과
2. `docs/experiments/index.md`의 실험 목록
3. Final Test가 실행된 경우 `docs/model_val.md`의 Test 비교표

재현되지 않은 수치를 문서에 기록하지 않는다. 목표 미달이어도 Test 결과로 임계값을 다시 선택하지 않는다.
